# 03 — Segmentação K-Means de Personas Comportamentais

**Objetivo:** agrupar os 175k VINs da rede oficial Ford em **4 personas** (`fiel`, `econômico`, `esquecido`, `abandono`) com base em features puramente comportamentais — sem rótulo, sem dado socioeconômico, sem `model_name`.

**Por que K-means cego (não classificação):**
- O dataset oficial v2 **não traz rótulo de persona** — não existe "verdade" para supervisão.
- Validação é feita via silhouette + cross-tabs label-free + nomeação heurística determinística.

**Regra crítica do [CLAUDE.md](../CLAUDE.md):** silhouette ≥ 0,35 é o piso aceitável para clustering comportamental (não exigir 0,7 — não é dado sintético).

**Implementação:** este notebook é um wrapper sobre [src/segmentation.py](../src/segmentation.py) (`sweep_k`, `fit`, `predict`, `save`). A lógica está toda lá; aqui está o relato acadêmico.

**Saídas:**
- `data/models/kmeans_segmentation.joblib`
- `data/models/kmeans_cluster_profiles.csv`
- `data/models/kmeans_metrics.json`
- `data/processed/vin_clusters.csv` (vin_hash → cluster_id, cluster_name)

Documentação completa em [MODEL_CARD_SEGMENTATION.md](../MODEL_CARD_SEGMENTATION.md).

## 0. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import load_features, KMEANS_FEATURES
from src import segmentation

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42
MODELS_DIR = REPO_ROOT / "data" / "models"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
FIG_DIR = REPO_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregar features comportamentais

A função `load_features()` aplica todas as regras de limpeza e derivação (clip de KM, parsing defensivo de datas, reference_date = `max(ServiceDate)` para reprodutibilidade) — é a **única** porta de entrada do pipeline.

In [ ]:
bundle = load_features()
df = bundle.df
print(f"Reference date (max ServiceDate): {bundle.reference_date.date()}")
print(f"VINs totais: {bundle.n_total:,}")
print(f"VINs trainable (tenure ≥ 365d): {bundle.n_trainable:,}")
print(f"\nFeatures usadas no K-means ({len(KMEANS_FEATURES)}):")
for f in KMEANS_FEATURES:
    print(f"  - {f}")
df.head()

## 2. EDA mínima das features comportamentais

Foco nas 5 features-chave que dirigem a separação: `days_since_last_service`, `tenure_days`, `events_count`, `dealers_distinct`, `gap_avg_days`. Espera-se distribuições bimodais (clientes ativos × clientes parados há muito tempo).

In [ ]:
df[KMEANS_FEATURES].describe().T[["mean", "std", "min", "50%", "max"]].round(2)

In [ ]:
key_feats = [
    "days_since_last_service", "tenure_days", "events_count",
    "dealers_distinct", "gap_avg_days", "km_per_month",
]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, key_feats):
    data = df[col].dropna()
    if data.max() > 1000:
        ax.hist(data, bins=60)
    else:
        ax.hist(data, bins=40)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("")
plt.suptitle("Distribuição das features comportamentais (175k VINs)", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_eda_kmeans_features.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Pré-processamento (interno ao `sweep_k`/`fit`)

O `StandardScaler` é aplicado dentro de `_prepare_matrix` em [src/segmentation.py:40](../src/segmentation.py#L40). Não há necessidade de scaling manual aqui — todas as features entram com escala neutra (z-score).

Features categóricas / identificadoras **NÃO** entram:
- `vin_hash` (identificador pseudonimizado) — privacidade.
- `model_name` (categorical alta cardinalidade) — viesaria por dominância RANGER/KA; queremos clusters de comportamento, não de modelo.
- `churned` (target supervisionado) — segmentação é **cega**, validação a posteriori.

## 4. Escolha de k — sweep silhouette + inertia

`segmentation.sweep_k(df, k_range=range(2,8))` roda K-means para cada k e calcula silhouette em sample de 10k (suficiente para estimativa estável; full dataset custaria minutos a mais sem ganho).

In [ ]:
sweep = segmentation.sweep_k(df, k_range=range(2, 8))
sweep

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(sweep["k"], sweep["silhouette"], "o-", color="tab:blue", label="silhouette")
ax1.set_xlabel("k (número de clusters)")
ax1.set_ylabel("Silhouette (sample 10k)", color="tab:blue")
ax1.axhline(0.35, ls="--", color="gray", alpha=0.7, label="piso aceitável (0.35)")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(sweep["k"], sweep["inertia"], "s--", color="tab:red", alpha=0.7, label="inertia")
ax2.set_ylabel("Inertia", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("Escolha de k — silhouette + elbow")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_kmeans_sweep.png", dpi=120, bbox_inches="tight")
plt.show()

**Decisão:** `k=4`.

Justificativa:
- Silhouette ≥ 0,35 (atende o piso do CLAUDE.md).
- Elbow visível em k=4 (inertia cai bem, ganhos marginais depois).
- 4 personas mapeiam diretamente para a nomenclatura de negócio do DOC 00 (`fiel`/`econômico`/`esquecido`/`abandono`) — não precisamos forçar 5 ou 6 clusters sem nome de negócio claro.

## 5. Treino do K-means final + nomeação das personas

`segmentation.fit(df, k=4)` faz:
1. Scaling + KMeans(`n_init=20`).
2. Calcula perfil médio por cluster (incluindo `churn_rate` post-hoc — só para validação).
3. Roda heurística `_name_clusters` para atribuir nomes (`fiel`, `econômico`, `esquecido`, `abandono`).
4. Recalcula silhouette no dataset completo (via sample 10k).

In [ ]:
artifacts = segmentation.fit(df, k=4, random_state=RANDOM_STATE)
print(json.dumps(artifacts.metrics, indent=2))
print("\nMapeamento cluster_id → nome:")
for cid, name in sorted(artifacts.cluster_names.items()):
    print(f"  cluster {cid} → {name}")

In [ ]:
profile_show = artifacts.cluster_profiles.copy()
profile_show = profile_show[["name", "size", "size_pct", "churn_rate"] + KMEANS_FEATURES]
profile_show.round(3)

## 6. Visualização das 4 personas

Dois gráficos:
1. **Barras de tamanho** — quantos VINs em cada persona, com % do total.
2. **Heatmap normalizado** das features-chave por persona — identifica visualmente o "DNA" de cada grupo.

In [ ]:
order = ["fiel", "economico", "esquecido", "abandono"]
colors = {"fiel": "#2ca02c", "economico": "#1f77b4", "esquecido": "#ff7f0e", "abandono": "#d62728"}

profile_named = artifacts.cluster_profiles.set_index("name").reindex(order)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(order, profile_named["size"], color=[colors[n] for n in order])
for i, (n, v) in enumerate(zip(order, profile_named["size"])):
    axes[0].text(i, v + 1000, f"{int(v):,}\n({profile_named.loc[n, 'size_pct']*100:.1f}%)", ha="center", fontsize=9)
axes[0].set_title("Tamanho de cada persona")
axes[0].set_ylabel("VINs")

axes[1].bar(order, profile_named["churn_rate"], color=[colors[n] for n in order])
for i, (n, v) in enumerate(zip(order, profile_named["churn_rate"])):
    axes[1].text(i, v + 0.02, f"{v*100:.1f}%", ha="center", fontsize=10)
axes[1].set_ylim(0, 1.1)
axes[1].axhline(df["churned"].mean(), ls="--", color="gray", label=f"média global ({df['churned'].mean()*100:.1f}%)")
axes[1].set_title("Taxa de churn (validação cruzada cega)")
axes[1].set_ylabel("Churn rate")
axes[1].legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "03_personas_size_churn.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
heat = profile_named[KMEANS_FEATURES].copy()
heat_norm = (heat - heat.min()) / (heat.max() - heat.min() + 1e-9)

fig, ax = plt.subplots(figsize=(11, 3.5))
sns.heatmap(heat_norm, annot=heat.round(1), fmt="", cmap="RdYlGn_r", cbar_kws={"label": "normalizado (0=min, 1=max)"}, ax=ax)
ax.set_title("DNA das personas — features normalizadas (rótulo = valor real)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_personas_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Cross-tabs label-free (sanity checks)

Se a segmentação faz sentido, devemos observar:
- `MaintenanceNumber` (número da revisão) varia consistentemente por cluster.
- `model_name` mostra distribuição diferente entre personas.
- `churned` (post-hoc) colide com `abandono`.

In [ ]:
scored = segmentation.predict(df, artifacts)
print("Tamanho médio por persona (deve bater com fit):")
print(scored["cluster_name"].value_counts(normalize=True).round(3))

In [ ]:
xt_model = (
    scored.groupby("cluster_name")["model_name"]
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)
top_models = scored["model_name"].value_counts().head(6).index
xt_model[top_models].round(3).reindex(order)

In [ ]:
xt_churn = scored.groupby("cluster_name").agg(
    n=("cluster_id", "size"),
    churn_rate=("churned", "mean"),
    days_since_mean=("days_since_last_service", "mean"),
    events_mean=("events_count", "mean"),
).reindex(order).round(2)
xt_churn

## 8. Estratégias de retenção por persona

Uma frase por segmento — entrada do relatório final.

### `fiel` (46% — 80,8k VINs, churn 45%)
Núcleo do negócio: ativos, dealer único, múltiplas revisões. **Estratégia:** programa de fidelidade (créditos por revisão, agendamento proativo da próxima manutenção, ofertas casadas com pneu/bateria no momento certo). O churn de 45% nesse grupo é *o mais valioso de combater* — perda de cliente caro.

### `econômico` (17% — 30,3k VINs, churn 11%)
Uso leve, 1 evento, sem sinal de abandono. **Estratégia:** lembrete educacional sobre a importância da segunda revisão (a famosa "revisão de 30k"), garantia estendida e mostrar valor da rede oficial vs. independente. Conversão potencial em `fiel`.

### `esquecido` (25% — 44,3k VINs, churn 58%)
Multi-evento mas drifting (gap_last 273d, dealer_share 59% — visita várias casas). **Estratégia:** **reativação urgente** — campanha personalizada de "sentimos sua falta", ofertas de troca de óleo/revisão a preço promocional, contato do dealer de maior afinidade. ROI alto: cliente conhece a rede, só precisa de gatilho.

### `abandono` (12% — 20,2k VINs, churn 99,98%)
Único evento, 1.700 dias sem voltar. **Estratégia:** **não investir em retenção massiva** — custo de reativação é alto e probabilidade baixa. Pode entrar em campanha de upgrade/recompra (modelo novo) ou ser usado como controle no A/B. Não bloquear contato, mas calibrar expectativa.

## 9. Persistência dos artefatos

Salva:
- `data/models/kmeans_segmentation.joblib` — scaler + modelo + nomes.
- `data/models/kmeans_cluster_profiles.csv` — perfis numéricos.
- `data/models/kmeans_metrics.json` — silhouette + inertia + n.
- `data/processed/vin_clusters.csv` — atribuição final por VIN (usado pelo `04_classification.ipynb` para validação cruzada com SHAP).

In [ ]:
segmentation.save(artifacts, MODELS_DIR)

vin_clusters = scored[["vin_hash", "cluster_id", "cluster_name", "cluster_distance"]]
vin_clusters.to_csv(PROCESSED_DIR / "vin_clusters.csv", index=False)

print(f"OK — escritos em {MODELS_DIR} e {PROCESSED_DIR / 'vin_clusters.csv'}")
print(f"   silhouette final: {artifacts.metrics['silhouette_10k_sample']:.4f}")

## Conclusão

- **k=4** atinge silhouette 0,40 (≥ piso 0,35 do CLAUDE.md).
- 4 personas com **nomes de negócio** (não "Cluster 0/1/2/3") — `fiel`, `econômico`, `esquecido`, `abandono`.
- Validação cruzada label-free: `abandono` colide com `churned=1` em 99,98% (sanity check ✓), enquanto `fiel` e `esquecido` têm churn intermediário — o classificador supervisionado V3 dá granularidade dentro desses dois grupos.
- **Estratégia de retenção priorizada:** `esquecido` > `fiel-em-risco` > `econômico` > `abandono`.

**Próximo notebook:** [04_classification.ipynb](04_classification.ipynb) — classificação binária (`churned`) com XGBoost V3 + SHAP médio por cluster como validação cruzada.

**Documentação completa:** [MODEL_CARD_SEGMENTATION.md](../MODEL_CARD_SEGMENTATION.md).